<a href="https://colab.research.google.com/github/saritrit66-collab/Final-Project/blob/main/%D7%91%D7%93%D7%99%D7%A7%D7%95%D7%AA%20%D7%A2%D7%9C%20%D7%9C%D7%99%D7%A0%D7%A7%D7%99%D7%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from tqdm import tqdm # בשביל סרגל ההתקדמות
import re

# --- 1. הפונקציה ששואבת את הנתונים מהאינטרנט ---
def scrape_engagement_from_url(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }

        # אם אין לינק חוקי, נחזיר 0
        if pd.isna(url) or not isinstance(url, str) or not url.startswith('http'):
            return pd.Series([0, 0])

        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            return pd.Series([0, 0])

        soup = BeautifulSoup(response.text, 'html.parser')
        likes = 0
        shares = 0

        # חיפוש לייקים
        like_element = soup.find(lambda tag: tag.name in ['span', 'div'] and
                                 tag.get('class') and
                                 any('like' in c.lower() or 'upvote' in c.lower() for c in tag.get('class')))
        if like_element:
            numbers = re.findall(r'\d+', like_element.text.replace(',', ''))
            if numbers:
                likes = int(numbers[0])

        # חיפוש שיתופים
        share_element = soup.find(lambda tag: tag.name in ['span', 'div'] and
                                  tag.get('class') and
                                  any('share' in c.lower() or 'retweet' in c.lower() for c in tag.get('class')))
        if share_element:
            numbers = re.findall(r'\d+', share_element.text.replace(',', ''))
            if numbers:
                shares = int(numbers[0])

        return pd.Series([likes, shares])
    except Exception as e:
        return pd.Series([0, 0])

# --- 2. טעינת הנתונים (עם פתרון לשגיאות הקידוד!) ---
print("טוען את הקובץ...")
# שימי לב: החליפי את 'your_file_name.csv' בשם הקובץ שלך!
file_name = 'your_file_name.csv'
df = pd.read_csv(file_name, encoding='utf-8', encoding_errors='ignore')

# --- 3. הרצת האלגוריתם ---
print(f"הקובץ נטען בהצלחה! מתחיל לסרוק {len(df)} שורות...")
tqdm.pandas(desc="Scraping URLs")

# האלגוריתם מופעל על עמודת ה-URL שבטבלה df (פותר את ה-NameError)
df[['Scraped_Likes', 'Scraped_Shares']] = df['URL'].progress_apply(scrape_engagement_from_url)

time.sleep(1) # השהייה קלה לסיום
print("הסריקה הסתיימה בהצלחה!")

# --- 4. שמירה והורדה (עם פתרון לעברית באקסל) ---
output_filename = 'Final_Scraped_Data.csv'
df.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"הנתונים נשמרו בקובץ: {output_filename}")

# מוריד את הקובץ ישירות למחשב
try:
    from google.colab import files
    files.download(output_filename)
except ImportError:
    pass

טוען את הקובץ...
הקובץ נטען בהצלחה! מתחיל לסרוק 19293 שורות...


Scraping URLs:  10%|▉         | 1899/19293 [1:07:58<14:43:11,  3.05s/it]